# SS3 Part B: Applying the SS2 Model to Real STADIOchoice Data

This notebook applies the final model selected in SS2 (XGBoost) to the real STADIOchoice data extract, as scoped in SS1's data request and now delivered by the client.

## 1. Dataset loading

The client extract (`datasets/raw/stadiochoice_ss3/`) contains six tables, matching the SS1 Part C data request exactly: subscription and billing, streaming viewing logs, cancellation and retention, customer support, content catalogue, and search queries. `household_id` is the join key across tables (except the content catalogue, keyed by `title_id`).

The primary entity is the household, one row per household in `table1_subscription_billing.csv`: **4,485 households**. The target is `is_active` (the client's own README confirms churn is equivalent to `is_active = FALSE`, i.e. `churn_date` populated). Of the 4,485 households, **849 have churned and 3,636 are still active, a churn rate of about 18.9%**. This is a meaningfully lower churn rate than SS2's Telco proxy dataset (26.5%), and closer to STADIOchoice's own reported 9.2% monthly streaming churn figure from the SS1 briefing pack, since this is an extract of the full subscriber base rather than a single month's worth of churn events.

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "datasets/raw/stadiochoice_ss3"
EXTRACT_DATE = pd.Timestamp("2026-08-31")  # per the client's own README

t1 = pd.read_csv(f"{DATA_DIR}/table1_subscription_billing.csv")
t2 = pd.read_csv(f"{DATA_DIR}/table2_viewing_logs.csv")
t3 = pd.read_csv(f"{DATA_DIR}/table3_cancellation_retention.csv")
t4 = pd.read_csv(f"{DATA_DIR}/table4_customer_support.csv")
t5 = pd.read_csv(f"{DATA_DIR}/table5_content_catalogue.csv")
t6 = pd.read_csv(f"{DATA_DIR}/table6_search_queries.csv")

for name, df in [("table1 subscription/billing", t1), ("table2 viewing logs", t2),
                 ("table3 cancellation/retention", t3), ("table4 customer support", t4),
                 ("table5 content catalogue", t5), ("table6 search queries", t6)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

churn_counts = t1["is_active"].value_counts()
print(f"\nTarget distribution (is_active): {churn_counts.to_dict()}")
print(f"Churn rate: {(~t1['is_active']).mean():.1%}")

table1 subscription/billing: 4485 rows, 12 columns
table2 viewing logs: 44525 rows, 7 columns
table3 cancellation/retention: 988 rows, 7 columns
table4 customer support: 3149 rows, 6 columns
table5 content catalogue: 640 rows, 6 columns
table6 search queries: 13587 rows, 4 columns

Target distribution (is_active): {True: 3636, False: 849}
Churn rate: 18.9%
